# Step 5 — What the trained network adds (GPU)

**So far (CPU):** along any fixed linear direction, the bifurcation diagram is a property of the data. MNIST balanced pairs show folds (43/45 along LDA), with fold size set by hump-width asymmetry.

**Two model-dependent questions:**

**(a) Fidelity.** Does a trained denoiser reproduce the data's fold along the LDA direction? We apply the Step-2 conditional-drift estimator with the real network and overlay the exact (data-only) diagram.

**(b) The network's own direction.** Along the top eigenvector of the denoiser Jacobian `J = ∂D/∂x` (the "soft mode"), which the data does not fix, what does the bifurcation look like? We track: the top eigenvalue vs σ (softening), the eigenvector's overlap with the LDA axis, and the drift diagram along it.

**Models:** one small EDM-style UNet per pair: (0,1) large fold, (3,8) overlap-heavy fold, (0,8) the one pitchfork-like LDA pair (our real-data control).

Kaggle: accelerator **GPU T4/P100**, Internet **on**. ~25–35 min total (3 × ~8 min training + analysis).

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, math
import matplotlib.pyplot as plt
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['figure.dpi'] = 110

ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X_ALL = (ds.data.numpy().astype(np.float32) / 127.5 - 1.0)      # (60000, 28, 28)
Y_ALL = ds.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]
SIGMA_DATA = 0.5

## Model: small EDM-preconditioned UNet

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1)
        self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return x + h

class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)   # 28->14
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1)                        # 14->28
        self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1)
        e = self.emb(e)
        h = self.inp(x); h1 = self.d1(h, e)
        h2 = self.d2(self.down(h1), e); h2 = self.mid(h2, e)
        h = self.u1(self.up(h2) + h1, e)
        return self.out(h)

def denoise(net, x, sigma):
    """EDM preconditioning: D(x, sigma) = c_skip x + c_out F(c_in x, c_noise). x: (B,1,28,28); sigma: (B,)"""
    s = sigma.view(-1, 1, 1, 1)
    c_skip = SIGMA_DATA**2 / (s**2 + SIGMA_DATA**2)
    c_out = s * SIGMA_DATA / torch.sqrt(s**2 + SIGMA_DATA**2)
    c_in = 1 / torch.sqrt(s**2 + SIGMA_DATA**2)
    c_noise = 0.25 * torch.log(sigma)
    return c_skip * x + c_out * net(c_in * x, c_noise)

def train(pair, epochs=20, bs=128, lr=2e-4):
    X = torch.from_numpy(X_ALL[np.isin(Y_ALL, pair)])[:, None].to(dev)
    net = UNet().to(dev); opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X); t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev); tot = 0
        for i in range(0, n - bs + 1, bs):
            x0 = X[perm[i:i+bs]]
            sigma = torch.exp(-1.2 + 1.2*torch.randn(bs, device=dev))          # EDM log-normal
            xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0)
            w = (sigma**2 + SIGMA_DATA**2) / (sigma*SIGMA_DATA)**2
            loss = (w.view(-1,1,1,1) * (denoise(net, xt, sigma) - x0)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if ep % 5 == 4 or ep == epochs-1:
            print(f"  pair {pair} epoch {ep+1}/{epochs} loss {tot/(n//bs):.4f}  ({time.time()-t0:.0f}s)")
    net.eval(); return net

NETS = {}
for pair in PAIRS:
    NETS[pair] = train(pair)
    torch.save(NETS[pair].state_dict(), f'/kaggle/working/unet_{pair[0]}{pair[1]}.pt')

## Shared analysis code (exact diagram, LDA direction, conditional-drift estimator)

In [ ]:
GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
TAUS = np.geomspace(0.1, 3.0, 36)

def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True)
    k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(np.convolve(h, k, mode='same'), 1e-300)

def stationary_points(p, merge=0.02):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    pts = [(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx]
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]:
                del pts[a:a+2]; ch = True; break
    return pts

def zero_crossings(cen, mean, cnt, min_count=150):
    ok = cnt >= min_count; c, f = cen[ok], mean[ok]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0:
            out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return out

def birth_jump(diag, taus):
    prev = None
    for tau in sorted(taus, reverse=True):
        mx = sorted(p for p, k in diag[tau] if k == 'max'); mn = [p for p, k in diag[tau] if k == 'min']
        if len(mx) >= 2 and prev is not None:
            c = mn[0] if mn else 0.5*(mx[0]+mx[-1]); return tau, abs(c-prev)
        if len(mx) == 1: prev = mx[0]
    return None

def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    mid = 0.5*(mu_a+mu_b); half = 0.5*((mu_a-mid)@u - (mu_b-mid)@u)
    return u.astype(np.float32), mid.astype(np.float32), float(half)

@torch.no_grad()
def net_drift_diagram(net, Xflat, u, mid, half, taus, n=8000, nbins=60):
    """Conditional drift along direction u, using the NETWORK denoiser. Returns {tau: zero crossings}.
    m = (x-mid).u/half ; score wrt m = half * (u . (D-x)/sigma^2)."""
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    out = {}
    for tau in taus:
        sigma = half*np.sqrt(tau)
        idx = rng.choice(len(Xflat), n); x0 = torch.from_numpy(Xflat[idx]).to(dev)
        xt = x0 + sigma*torch.randn_like(x0)
        D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1)
                       for xb in xt.split(1000)])
        m = ((xt-midt) @ ut / half).cpu().numpy()
        sm = (half * ((D-xt) @ ut) / sigma**2).cpu().numpy()
        lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1)
        b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
        cnt = np.bincount(b, minlength=nbins); mean = np.bincount(b, weights=sm, minlength=nbins)/np.maximum(cnt, 1)
        out[tau] = zero_crossings(0.5*(edges[:-1]+edges[1:]), mean, cnt)
    return out

def exact_diagram(Xflat, u, mid, half, taus):
    m0 = (Xflat-mid) @ u / half
    return {tau: stationary_points(smooth_hist(m0, tau)) for tau in taus}

## (a) Fidelity: network drift diagram vs exact data diagram, along LDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, pair in zip(axes, PAIRS):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); Xflat = np.concatenate([Xa[:nmin], Xb[:nmin]])     # balanced
    u, mid, half = lda_direction(Xa, Xb)
    ex = exact_diagram(Xflat, u, mid, half, TAUS)
    nd = net_drift_diagram(NETS[pair], Xflat, u, mid, half, TAUS)
    bj_e, bj_n = birth_jump(ex, TAUS), birth_jump(nd, TAUS)
    for tau in TAUS:
        for p, k in ex[tau]: ax.plot(tau, p, '_', ms=9, color='k', alpha=0.4)
        for p, k in nd[tau]: ax.plot(tau, p, 'o', ms=3, color='C0' if k == 'max' else 'C3')
    f = lambda bj: f"tau={bj[0]:.2f}, jump={bj[1]:.2f}" if bj else "none"
    ax.set_xscale('log'); ax.set_title(f"{pair} LDA  exact: {f(bj_e)}\nnetwork: {f(bj_n)}", fontsize=9)
    ax.set_xlabel('tau'); ax.axhline(0, color='k', lw=0.4)
    print(f"{pair}: exact {f(bj_e)} | network {f(bj_n)}")
axes[0].set_ylabel('fixed point m* (grey = exact/data, colored = network)')
plt.tight_layout(); plt.show()

## (b) The network's own soft mode

Power iteration on the batch-averaged Jacobian of `D` (finite-difference JVP). We report the top eigenvalue λ(σ), the overlap |v·u_LDA|, and the drift diagram along v.

In [ ]:
@torch.no_grad()
def soft_mode(net, Xflat, sigma, n=256, iters=15, eps=1e-2):
    idx = rng.choice(len(Xflat), n); x0 = torch.from_numpy(Xflat[idx]).to(dev)
    xt = (x0 + sigma*torch.randn_like(x0)).view(-1,1,28,28); s = torch.full((n,), sigma, device=dev)
    v = torch.randn(784, device=dev); v /= v.norm()
    for _ in range(iters):
        vv = v.view(1,1,28,28)
        Jv = ((denoise(net, xt+eps*vv, s) - denoise(net, xt-eps*vv, s))/(2*eps)).view(n, -1).mean(0)
        lam = float(Jv @ v); v = Jv / Jv.norm()
    return v.cpu().numpy(), lam

SIG_GRID = np.geomspace(0.15, 3.0, 20)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, pair in enumerate(PAIRS):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); Xflat = np.concatenate([Xa[:nmin], Xb[:nmin]])
    u, mid, half = lda_direction(Xa, Xb)
    lams, ovl, vs = [], [], []
    for sigma in SIG_GRID:
        v, lam = soft_mode(NETS[pair], Xflat, float(sigma)); lams.append(lam); ovl.append(abs(v @ u)); vs.append(v)
    axes[0, j].plot(SIG_GRID**2/half**2, lams, 'o-', label='top eigenvalue of J')
    axes[0, j].plot(SIG_GRID**2/half**2, ovl, 's-', label='|v . u_LDA|')
    axes[0, j].set_xscale('log'); axes[0, j].set_xlabel('tau (sigma^2/half^2)'); axes[0, j].set_title(f'{pair}: soft mode'); axes[0, j].legend(fontsize=8)
    # drift diagram along the soft mode measured at the sigma nearest the exact bifurcation
    ex = exact_diagram(Xflat, u, mid, half, TAUS); bj = birth_jump(ex, TAUS)
    tau_b = bj[0] if bj else 0.8
    k = int(np.argmin(np.abs(SIG_GRID**2/half**2 - tau_b))); v = vs[k].astype(np.float32)
    # same sigma grid as the LDA analysis (tau = sigma^2/half^2) so the two diagrams are comparable
    nd_v = net_drift_diagram(NETS[pair], Xflat, v, mid, half, TAUS)
    for tau in TAUS:
        for p, kk in nd_v[tau]: axes[1, j].plot(tau, p, 'o', ms=3, color='C0' if kk == 'max' else 'C3')
    bjv = birth_jump(nd_v, TAUS)
    axes[1, j].set_xscale('log'); axes[1, j].set_xlabel('tau'); axes[1, j].axhline(0, color='k', lw=0.4)
    axes[1, j].set_title(f'{pair}: drift along soft mode v(tau={tau_b:.2f}), overlap {ovl[k]:.2f}\n' + (f'jump={bjv[1]:.2f} at tau={bjv[0]:.2f}' if bjv else 'never bimodal'), fontsize=9)
    print(f"{pair}: top eigenvalue range {min(lams):.3f}..{max(lams):.3f}; overlap at tau_bif {ovl[k]:.2f}; soft-mode diagram: {bjv}")
plt.tight_layout(); plt.show()

## What to report back

1. Training losses (sanity: they should fall to roughly 0.05–0.15 in EDM-weighted units).
2. **(a)** For each pair: exact vs network `tau_bif` and `jump`. Does the learned score preserve the fold, shift it, or close it? The (0,8) pair is the control: exact jump ≈ 0.03.
3. **(b)** The eigenvalue and overlap curves, and the soft-mode diagrams. Two things matter: whether λ approaches 1 (softening) near the LDA bifurcation, and whether the soft mode is aligned with LDA (overlap near 1) or points somewhere else.